# Blunder prediction overnight run

This notebook is designed to run unattended for several hours.

It adds checkpointing and model reuse to the working pawn-loss pipeline:

1. Every trained model is saved immediately.
2. If a saved model exists, reruns load it instead of retraining.
3. Optuna studies are stored in SQLite and can resume after interruption.
4. Metrics, plots, model scores, and permutation importance are exported.

Naming convention used here:

- files use underscores, for example `summary_test.csv`
- directories use hyphens, for example `overnight-runs`

Output directory:

```text
scratch/overnight-runs/blunder-prediction-overnight/
```


## 0. Dependencies

Recommended install:

```bash
uv add xgboost lightgbm torch optuna joblib
```


In [1]:
import os

# Mac/PyTorch safety settings. Keep these before importing torch.
FORCE_TORCH_CPU = True
TORCH_NUM_THREADS = 1

os.environ["OMP_NUM_THREADS"] = str(TORCH_NUM_THREADS)
os.environ["MKL_NUM_THREADS"] = str(TORCH_NUM_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(TORCH_NUM_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(TORCH_NUM_THREADS)


In [2]:
from pathlib import Path
import importlib.util
import json
import random
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
  ConfusionMatrixDisplay,
  PrecisionRecallDisplay,
  RocCurveDisplay,
  average_precision_score,
  balanced_accuracy_score,
  classification_report,
  confusion_matrix,
  f1_score,
  roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Main data-size control.
# Set to None for all games. Safer overnight values: 150_000 to 250_000.
N_GAMES_SAMPLE = 200_000
RANDOM_GAME_SAMPLE = True

# Pawn-loss blunder definition.
MISTAKE_PAWN_LOSS = 1.0
BLUNDER_PAWN_LOSS = 2.0
HORIZON_OWN_MOVES = 5
MAX_ABS_EVAL_PAWNS = 50.0

# Reuse/saving behavior.
REUSE_SAVED_MODELS = True
FORCE_RETRAIN_MODELS = False
SAVE_MODEL_SCORES = True

# Hyperparameter optimization.
RUN_HYPERPARAM_OPT = True
OPTUNA_TRIALS_XGB = 30
OPTUNA_TRIALS_LGBM = 30
OPTUNA_TRIALS_MLP = 12
OPTUNA_TRIALS_FT = 6
OPTUNA_TIMEOUT_SECONDS = None

# PyTorch controls.
NN_TRAIN_MAX_ROWS = 500_000
NN_VALID_INTERNAL_MAX_ROWS = 50_000
NN_MAX_EPOCHS = 12
NN_BATCH_SIZE = 4096
NN_PATIENCE = 4
NN_LEARNING_RATE = 1e-3
NN_PROGRESS_EVERY_N_BATCHES = 20

RUN_MLP = True
RUN_FT_TRANSFORMER = True

# Keep transformer smaller than MLP.
FT_TRAIN_MAX_ROWS = 100_000
FT_BATCH_SIZE = 2048
FT_MAX_EPOCHS = 5
FT_PATIENCE = 2

# Permutation importance.
RUN_PERMUTATION_IMPORTANCE = True
PERM_IMPORTANCE_MAX_ROWS = 30_000
PERM_IMPORTANCE_REPEATS = 3

RUN_BASELINES = True
RUN_XGBOOST = True
RUN_LIGHTGBM = True

pd.set_option("display.max_columns", 150)


In [3]:
def has_package(name):
  return importlib.util.find_spec(name) is not None


HAS_XGBOOST = has_package("xgboost")
HAS_LIGHTGBM = has_package("lightgbm")
HAS_TORCH = has_package("torch")
HAS_OPTUNA = has_package("optuna")

print("Optional package availability")
print("xgboost :", HAS_XGBOOST)
print("lightgbm:", HAS_LIGHTGBM)
print("torch   :", HAS_TORCH)
print("optuna  :", HAS_OPTUNA)


Optional package availability
xgboost : True
lightgbm: True
torch   : True
optuna  : True


## 1. Project paths


In [4]:
def find_project_root(start=None):
  if start is None:
    start = Path.cwd()

  start = Path(start).resolve()

  for path in [start, *start.parents]:
    has_pyproject = (path / "pyproject.toml").exists()
    has_data = (path / "data").exists()

    if has_pyproject and has_data:
      return path

  raise FileNotFoundError(
    "Could not find project root. Run from the project root "
    "or from a subdirectory such as scratch/."
  )


ROOT = find_project_root()
DATA_DIR = ROOT / "data" / "processed" / "lichess-2017-05-eval-all"

GAMES_PATH = DATA_DIR / "games.parquet"
PLIES_PATH = DATA_DIR / "plies.parquet"
FEATURES_PATH = DATA_DIR / "features.parquet"
DICT_PATH = DATA_DIR / "feature_dictionary.csv"

RUN_DIR = (
  ROOT
  / "scratch"
  / "overnight-runs"
  / "blunder-prediction-overnight"
)
MODEL_DIR = RUN_DIR / "models"
SCORE_DIR = RUN_DIR / "scores"
METRIC_DIR = RUN_DIR / "metrics"
PLOT_DIR = RUN_DIR / "plots"
IMPORTANCE_DIR = RUN_DIR / "importance"
STUDY_DIR = RUN_DIR / "studies"

for path in [
  RUN_DIR,
  MODEL_DIR,
  SCORE_DIR,
  METRIC_DIR,
  PLOT_DIR,
  IMPORTANCE_DIR,
  STUDY_DIR,
]:
  path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Run dir:      {RUN_DIR}")


Project root: /Users/asudler/Desktop/ucph/coursework/2526-block4/aml/final-project/root/workspace/aidan_sandbox/blunder
Data dir:     /Users/asudler/Desktop/ucph/coursework/2526-block4/aml/final-project/root/workspace/aidan_sandbox/blunder/data/processed/lichess-2017-05-eval-all
Run dir:      /Users/asudler/Desktop/ucph/coursework/2526-block4/aml/final-project/root/workspace/aidan_sandbox/blunder/scratch/overnight-runs/blunder-prediction-overnight


In [5]:
run_config = {
  "N_GAMES_SAMPLE": N_GAMES_SAMPLE,
  "RANDOM_GAME_SAMPLE": RANDOM_GAME_SAMPLE,
  "MISTAKE_PAWN_LOSS": MISTAKE_PAWN_LOSS,
  "BLUNDER_PAWN_LOSS": BLUNDER_PAWN_LOSS,
  "HORIZON_OWN_MOVES": HORIZON_OWN_MOVES,
  "MAX_ABS_EVAL_PAWNS": MAX_ABS_EVAL_PAWNS,
  "REUSE_SAVED_MODELS": REUSE_SAVED_MODELS,
  "FORCE_RETRAIN_MODELS": FORCE_RETRAIN_MODELS,
  "RUN_HYPERPARAM_OPT": RUN_HYPERPARAM_OPT,
  "OPTUNA_TRIALS_XGB": OPTUNA_TRIALS_XGB,
  "OPTUNA_TRIALS_LGBM": OPTUNA_TRIALS_LGBM,
  "OPTUNA_TRIALS_MLP": OPTUNA_TRIALS_MLP,
  "OPTUNA_TRIALS_FT": OPTUNA_TRIALS_FT,
  "NN_TRAIN_MAX_ROWS": NN_TRAIN_MAX_ROWS,
  "FT_TRAIN_MAX_ROWS": FT_TRAIN_MAX_ROWS,
  "FORCE_TORCH_CPU": FORCE_TORCH_CPU,
  "TORCH_NUM_THREADS": TORCH_NUM_THREADS,
  "PERM_IMPORTANCE_MAX_ROWS": PERM_IMPORTANCE_MAX_ROWS,
  "PERM_IMPORTANCE_REPEATS": PERM_IMPORTANCE_REPEATS,
}

with open(RUN_DIR / "run_config.json", "w") as f:
  json.dump(run_config, f, indent=2)


## 2. Load data and detect columns


In [6]:
load_start = time.time()

games = pd.read_parquet(GAMES_PATH)
plies = pd.read_parquet(PLIES_PATH)
features = pd.read_parquet(FEATURES_PATH)

if DICT_PATH.exists():
  feature_dict = pd.read_csv(DICT_PATH)
else:
  feature_dict = pd.DataFrame()

print("games:", games.shape)
print("plies:", plies.shape)
print("features:", features.shape)
print("feature_dict:", feature_dict.shape)
print(f"Load time: {time.time() - load_start:.1f}s")


games: (957369, 17)
plies: (61361243, 39)
features: (957369, 80)
feature_dict: (6, 5)
Load time: 53.7s


In [7]:
COLUMN_OVERRIDES = {
  # "game_id": "game_index",
  # "ply_index": "ply",
  # "eval_pawns": "eval_pawns",
  # "turn": "side",
}


def pick_column(df, candidates, required=True, role="column"):
  for key in candidates:
    if key in COLUMN_OVERRIDES:
      override = COLUMN_OVERRIDES[key]
      if override in df.columns:
        return override

  for name in candidates:
    if name in df.columns:
      return name

  if required:
    cols = ", ".join(candidates)
    raise KeyError(f"Could not find {role}. Tried: {cols}")

  return None


def pick_contains(df, include, exclude=None, required=True, role="column"):
  if exclude is None:
    exclude = []

  include = [x.lower() for x in include]
  exclude = [x.lower() for x in exclude]

  matches = []
  for col in df.columns:
    low = col.lower()
    has_all = all(x in low for x in include)
    has_excl = any(x in low for x in exclude)
    if has_all and not has_excl:
      matches.append(col)

  if matches:
    return matches[0]

  if required:
    raise KeyError(f"Could not find {role} containing {include}")

  return None


game_id_col = pick_column(
  plies,
  ["game_id", "game_index", "game_idx", "id"],
  role="game id column",
)

ply_col = pick_column(
  plies,
  ["ply", "ply_index", "ply_number", "move_ply"],
  role="ply index column",
)

eval_col = pick_column(
  plies,
  [
    "eval_pawns",
    "eval",
    "score",
    "engine_eval",
    "stockfish_eval",
    "stockfish_eval_pawns",
  ],
  required=False,
  role="engine eval pawn column",
)

if eval_col is None:
  eval_col = pick_contains(
    plies,
    ["eval"],
    exclude=["mate", "raw", "comment"],
    role="engine eval pawn column",
  )

turn_col = pick_column(
  plies,
  ["turn", "side", "color", "player_color", "mover"],
  required=False,
  role="side-to-move column",
)

print("Detected columns")
print("game_id_col:", game_id_col)
print("ply_col:    ", ply_col)
print("eval_col:   ", eval_col)
print("turn_col:   ", turn_col)


Detected columns
game_id_col: game_index
ply_col:     ply
eval_col:    eval_pawns
turn_col:    side


## 3. Sample games, construct labels, and build features


In [8]:
def sample_games(df, n_games, random_sample=True, seed=RANDOM_STATE):
  if n_games is None:
    return df.copy()

  game_ids = df[game_id_col].drop_duplicates()

  if n_games >= len(game_ids):
    return df.copy()

  if random_sample:
    selected_ids = game_ids.sample(n_games, random_state=seed)
  else:
    selected_ids = game_ids.head(n_games)

  sampled = df[df[game_id_col].isin(selected_ids)].copy()
  sampled = sampled.sort_values([game_id_col, ply_col]).reset_index(drop=True)

  return sampled


sample_start = time.time()

plies = sample_games(
  plies,
  N_GAMES_SAMPLE,
  random_sample=RANDOM_GAME_SAMPLE,
)

sampled_game_ids = set(plies[game_id_col].drop_duplicates())

if game_id_col in games.columns:
  games = games[games[game_id_col].isin(sampled_game_ids)].copy()

if game_id_col in features.columns:
  features = features[features[game_id_col].isin(sampled_game_ids)].copy()

print("After sampling")
print("games:", games.shape)
print("plies:", plies.shape)
print("features:", features.shape)
print("unique games in plies:", plies[game_id_col].nunique())
print(f"Sampling time: {time.time() - sample_start:.1f}s")


After sampling
games: (200000, 17)
plies: (12821357, 39)
features: (200000, 80)
unique games in plies: 200000
Sampling time: 61.1s


In [9]:
def normalize_turn_value(x):
  if pd.isna(x):
    return np.nan

  value = str(x).strip().lower()

  if value in {"w", "white", "1", "true"}:
    return "white"

  if value in {"b", "black", "0", "false"}:
    return "black"

  return value


def add_basic_ply_columns(df):
  df = df.copy()
  df = df.sort_values([game_id_col, ply_col]).reset_index(drop=True)

  df["eval_pawns"] = pd.to_numeric(df[eval_col], errors="coerce")
  df["eval_pawns"] = df["eval_pawns"].clip(
    -MAX_ABS_EVAL_PAWNS,
    MAX_ABS_EVAL_PAWNS,
  )

  if turn_col is not None:
    df["side_to_move"] = df[turn_col].map(normalize_turn_value)
  else:
    df["side_to_move"] = np.where(df[ply_col] % 2 == 1, "white", "black")

  df["mover"] = df["side_to_move"]
  df["ply_number"] = pd.to_numeric(df[ply_col], errors="coerce")
  df["fullmove_number"] = ((df["ply_number"] + 1) // 2).astype("Int64")

  return df


def add_pawn_loss_labels(df):
  df = df.copy()
  group = df.groupby(game_id_col, sort=False)

  df["eval_after_pawns"] = group["eval_pawns"].shift(-1)
  df["eval_delta_pawns"] = df["eval_after_pawns"] - df["eval_pawns"]

  white_loss = -df["eval_delta_pawns"]
  black_loss = df["eval_delta_pawns"]

  df["pawn_loss"] = np.where(
    df["mover"].eq("white"),
    white_loss,
    black_loss,
  )

  df["pawn_loss"] = df["pawn_loss"].clip(lower=0)

  df["is_mistake"] = (
    df["pawn_loss"].ge(MISTAKE_PAWN_LOSS).astype(int)
  )

  df["is_blunder"] = (
    df["pawn_loss"].ge(BLUNDER_PAWN_LOSS).astype(int)
  )

  return df


def add_future_blunder_target_fast(df, horizon_own_moves):
  df = df.copy()
  df = df.sort_values([game_id_col, "mover", ply_col])

  keys = [game_id_col, "mover"]
  group = df.groupby(keys, sort=False)["is_blunder"]

  future_count = pd.Series(0, index=df.index, dtype=float)

  for step in range(1, horizon_own_moves + 1):
    future_count += group.shift(-step).fillna(0)

  df["future_blunder_count"] = future_count
  df["will_blunder_soon"] = (
    df["future_blunder_count"].gt(0).astype(int)
  )

  df = df.sort_values([game_id_col, ply_col]).reset_index(drop=True)

  return df


target_start = time.time()

plies_ml = add_basic_ply_columns(plies)
plies_ml = add_pawn_loss_labels(plies_ml)
plies_ml = add_future_blunder_target_fast(
  plies_ml,
  HORIZON_OWN_MOVES,
)

print("Move-level mistake/blunder fractions")
display(plies_ml[["is_mistake", "is_blunder"]].mean().to_frame("fraction"))

print("Future-blunder target counts")
display(plies_ml["will_blunder_soon"].value_counts(dropna=False))
display(plies_ml["will_blunder_soon"].value_counts(
  normalize=True,
  dropna=False,
))

print(f"Target construction time: {time.time() - target_start:.1f}s")


Move-level mistake/blunder fractions


,fraction
is_mistake,0.001601
is_blunder,0.000836


Future-blunder target counts


will_blunder_soon
0    12770141
1       51216
Name: count, dtype: int64

will_blunder_soon
0    0.996005
1    0.003995
Name: proportion, dtype: float64

Target construction time: 21.2s


In [10]:
def add_rolling_features(df):
  df = df.copy()
  df = df.sort_values([game_id_col, ply_col]).reset_index(drop=True)

  group = df.groupby(game_id_col, sort=False)

  df["abs_eval_pawns"] = df["eval_pawns"].abs()
  df["eval_sign"] = np.sign(df["eval_pawns"]).astype(float)

  df["eval_prev_1"] = group["eval_pawns"].shift(1)
  df["eval_prev_2"] = group["eval_pawns"].shift(2)
  df["eval_prev_4"] = group["eval_pawns"].shift(4)

  df["eval_change_prev_1"] = df["eval_pawns"] - df["eval_prev_1"]
  df["eval_change_prev_2"] = df["eval_pawns"] - df["eval_prev_2"]
  df["eval_change_prev_4"] = df["eval_pawns"] - df["eval_prev_4"]

  df["abs_eval_change_prev_1"] = df["eval_change_prev_1"].abs()
  df["abs_eval_change_prev_2"] = df["eval_change_prev_2"].abs()
  df["abs_eval_change_prev_4"] = df["eval_change_prev_4"].abs()

  df["game_ply_fraction"] = group.cumcount()
  max_ply = group["game_ply_fraction"].transform("max").replace(0, np.nan)
  df["game_ply_fraction"] = df["game_ply_fraction"] / max_ply

  df["phase"] = pd.cut(
    df["fullmove_number"].astype(float),
    bins=[0, 10, 25, 10_000],
    labels=["opening", "middlegame", "endgame"],
    include_lowest=True,
  )

  return df


def add_player_history_features(df):
  df = df.copy()
  df = df.sort_values([game_id_col, ply_col]).reset_index(drop=True)

  keys = [game_id_col, "mover"]
  group = df.groupby(keys, sort=False)

  prev_blunder = group["is_blunder"].shift(1).fillna(0)
  prev_mistake = group["is_mistake"].shift(1).fillna(0)
  prev_pawn_loss = group["pawn_loss"].shift(1)

  hist_keys = [df[game_id_col], df["mover"]]

  df["prev_own_blunders"] = (
    prev_blunder
    .groupby(hist_keys, sort=False)
    .cumsum()
  )

  df["prev_own_mistakes"] = (
    prev_mistake
    .groupby(hist_keys, sort=False)
    .cumsum()
  )

  df["prev_own_pawn_loss"] = prev_pawn_loss

  for window in [3, 5, 10]:
    shifted_loss = group["pawn_loss"].shift(1)

    rolled = (
      shifted_loss
      .groupby(hist_keys, sort=False)
      .rolling(window, min_periods=1)
      .mean()
    )

    df[f"prev_own_pawn_loss_mean_{window}"] = (
      rolled.reset_index(level=[0, 1], drop=True)
    )

  return df


feature_start = time.time()

plies_ml = add_rolling_features(plies_ml)
plies_ml = add_player_history_features(plies_ml)

candidate_features = [
  "eval_pawns",
  "abs_eval_pawns",
  "eval_sign",
  "fullmove_number",
  "game_ply_fraction",
  "eval_prev_1",
  "eval_prev_2",
  "eval_prev_4",
  "eval_change_prev_1",
  "eval_change_prev_2",
  "eval_change_prev_4",
  "abs_eval_change_prev_1",
  "abs_eval_change_prev_2",
  "abs_eval_change_prev_4",
  "prev_own_blunders",
  "prev_own_mistakes",
  "prev_own_pawn_loss",
  "prev_own_pawn_loss_mean_3",
  "prev_own_pawn_loss_mean_5",
  "prev_own_pawn_loss_mean_10",
  "mover",
  "phase",
]

feature_cols = [c for c in candidate_features if c in plies_ml.columns]

print("Feature columns:")
for col in feature_cols:
  print(" ", col)

print(f"Feature construction time: {time.time() - feature_start:.1f}s")


Feature columns:
  eval_pawns
  abs_eval_pawns
  eval_sign
  fullmove_number
  game_ply_fraction
  eval_prev_1
  eval_prev_2
  eval_prev_4
  eval_change_prev_1
  eval_change_prev_2
  eval_change_prev_4
  abs_eval_change_prev_1
  abs_eval_change_prev_2
  abs_eval_change_prev_4
  prev_own_blunders
  prev_own_mistakes
  prev_own_pawn_loss
  prev_own_pawn_loss_mean_3
  prev_own_pawn_loss_mean_5
  prev_own_pawn_loss_mean_10
  mover
  phase
Feature construction time: 25.3s


## 4. Train/validation/test split


In [11]:
model_df = plies_ml.dropna(subset=["will_blunder_soon"]).copy()

group = model_df.groupby(game_id_col, sort=False)
max_ply = group[ply_col].transform("max")
min_future_gap = 2 * HORIZON_OWN_MOVES
model_df = model_df[model_df[ply_col] <= max_ply - min_future_gap].copy()

X = model_df[feature_cols]
y = model_df["will_blunder_soon"].astype(int)
groups = model_df[game_id_col]

print("Model rows:", len(model_df))
print("Unique games:", groups.nunique())
print("Positive target rate:", y.mean())

display(y.value_counts(dropna=False))
display(y.value_counts(normalize=True, dropna=False))


Model rows: 10821731
Unique games: 199786
Positive target rate: 0.0041593161020173205


will_blunder_soon
0    10776720
1       45011
Name: count, dtype: int64

will_blunder_soon
0    0.995841
1    0.004159
Name: proportion, dtype: float64

In [12]:
def stratified_game_train_valid_test_split(
  X,
  y,
  groups,
  seed=RANDOM_STATE,
):
  game_df = (
    pd.DataFrame({
      "game_id": groups,
      "target": y,
    })
    .groupby("game_id", as_index=False)["target"]
    .max()
  )

  if game_df["target"].nunique() < 2:
    raise ValueError(
      "Only one game-level target class exists. Lower "
      "BLUNDER_PAWN_LOSS or increase N_GAMES_SAMPLE."
    )

  train_games, temp_games = train_test_split(
    game_df,
    train_size=0.70,
    random_state=seed,
    stratify=game_df["target"],
  )

  valid_games, test_games = train_test_split(
    temp_games,
    train_size=0.50,
    random_state=seed,
    stratify=temp_games["target"],
  )

  train_ids = set(train_games["game_id"])
  valid_ids = set(valid_games["game_id"])
  test_ids = set(test_games["game_id"])

  train_mask = groups.isin(train_ids)
  valid_mask = groups.isin(valid_ids)
  test_mask = groups.isin(test_ids)

  return (
    X.loc[train_mask],
    X.loc[valid_mask],
    X.loc[test_mask],
    y.loc[train_mask],
    y.loc[valid_mask],
    y.loc[test_mask],
    groups.loc[train_mask],
    groups.loc[valid_mask],
    groups.loc[test_mask],
  )


split = stratified_game_train_valid_test_split(X, y, groups)

(
  X_train,
  X_valid,
  X_test,
  y_train,
  y_valid,
  y_test,
  g_train,
  g_valid,
  g_test,
) = split

print("train:", X_train.shape, y_train.mean(), g_train.nunique())
print("valid:", X_valid.shape, y_valid.mean(), g_valid.nunique())
print("test: ", X_test.shape, y_test.mean(), g_test.nunique())


train: (7574112, 22) 0.004169465674655986 139850
valid: (1622199, 22) 0.00413019611034158 29968
test:  (1625420, 22) 0.004141083535332406 29968


In [13]:
numeric_cols = [
  c for c in feature_cols
  if pd.api.types.is_numeric_dtype(X_train[c])
]

categorical_cols = [
  c for c in feature_cols
  if c not in numeric_cols
]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

numeric_preprocess_scaled = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
  ("scaler", StandardScaler()),
])

numeric_preprocess_unscaled = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
])

categorical_preprocess = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_tree = ColumnTransformer([
  ("num", numeric_preprocess_unscaled, numeric_cols),
  ("cat", categorical_preprocess, categorical_cols),
])

preprocess_nn = ColumnTransformer([
  ("num", numeric_preprocess_scaled, numeric_cols),
  ("cat", categorical_preprocess, categorical_cols),
])

preprocess_linear = ColumnTransformer([
  ("num", numeric_preprocess_scaled, numeric_cols),
  ("cat", categorical_preprocess, categorical_cols),
])


Numeric columns: ['eval_pawns', 'abs_eval_pawns', 'eval_sign', 'fullmove_number', 'game_ply_fraction', 'eval_prev_1', 'eval_prev_2', 'eval_prev_4', 'eval_change_prev_1', 'eval_change_prev_2', 'eval_change_prev_4', 'abs_eval_change_prev_1', 'abs_eval_change_prev_2', 'abs_eval_change_prev_4', 'prev_own_blunders', 'prev_own_mistakes', 'prev_own_pawn_loss', 'prev_own_pawn_loss_mean_3', 'prev_own_pawn_loss_mean_5', 'prev_own_pawn_loss_mean_10']
Categorical columns: ['mover', 'phase']


## 5. Saving, loading, and evaluation utilities


In [14]:
def model_path(name):
  return MODEL_DIR / f"{name}.joblib"


def metadata_path(name):
  return MODEL_DIR / f"{name}_metadata.json"


def valid_score_path(name):
  return SCORE_DIR / f"{name}_valid_score.npy"


def test_score_path(name):
  return SCORE_DIR / f"{name}_test_score.npy"


def classification_report_path(name):
  return METRIC_DIR / f"{name}_classification_report.txt"


def get_score(model, X_eval):
  if hasattr(model, "predict_proba"):
    proba = model.predict_proba(X_eval)

    if proba.shape[1] == 1:
      only_class = model.classes_[0]
      if only_class == 1:
        return np.ones(len(X_eval))
      return np.zeros(len(X_eval))

    positive_index = list(model.classes_).index(1)
    return proba[:, positive_index]

  if hasattr(model, "decision_function"):
    score = model.decision_function(X_eval)
    return 1 / (1 + np.exp(-score))

  return model.predict(X_eval)


def find_best_threshold(y_true, score, metric="f1"):
  rows = []

  for threshold in np.linspace(0.01, 0.99, 99):
    pred = (score >= threshold).astype(int)

    rows.append({
      "threshold": threshold,
      "f1": f1_score(y_true, pred),
      "balanced_acc": balanced_accuracy_score(y_true, pred),
      "pred_positive_rate": pred.mean(),
    })

  threshold_df = pd.DataFrame(rows)
  best = threshold_df.sort_values(metric, ascending=False).iloc[0]

  return float(best["threshold"]), threshold_df


def evaluate_scores(name, y_true, score, threshold=0.5):
  pred = (score >= threshold).astype(int)

  return {
    "model": name,
    "threshold": threshold,
    "roc_auc": roc_auc_score(y_true, score),
    "avg_precision": average_precision_score(y_true, score),
    "balanced_acc": balanced_accuracy_score(y_true, pred),
    "f1": f1_score(y_true, pred),
    "pred_positive_rate": pred.mean(),
    "true_positive_rate": y_true.mean(),
  }


def fit_and_score_sklearn_model(name, model):
  start = time.time()
  model.fit(X_train, y_train)
  fit_seconds = time.time() - start

  valid_score = get_score(model, X_valid)
  threshold, threshold_df = find_best_threshold(y_valid, valid_score)

  valid_metrics_05 = evaluate_scores(
    name,
    y_valid,
    valid_score,
    threshold=0.5,
  )

  valid_metrics_opt = evaluate_scores(
    name,
    y_valid,
    valid_score,
    threshold=threshold,
  )

  test_score = get_score(model, X_test)

  test_metrics_opt = evaluate_scores(
    name,
    y_test,
    test_score,
    threshold=threshold,
  )

  for row in [valid_metrics_05, valid_metrics_opt, test_metrics_opt]:
    row["fit_seconds"] = fit_seconds

  return {
    "name": name,
    "model": model,
    "threshold": threshold,
    "threshold_df": threshold_df,
    "valid_score": valid_score,
    "test_score": test_score,
    "valid_metrics_05": valid_metrics_05,
    "valid_metrics_opt": valid_metrics_opt,
    "test_metrics_opt": test_metrics_opt,
    "fit_seconds": fit_seconds,
  }


def save_artifact(artifact):
  name = artifact["name"]
  model = artifact["model"]

  try:
    torch_model = model.named_steps.get("model")
    if hasattr(torch_model, "model_"):
      torch_model.model_.to("cpu")
  except Exception:
    pass

  joblib.dump(model, model_path(name))

  if SAVE_MODEL_SCORES:
    np.save(valid_score_path(name), artifact["valid_score"])
    np.save(test_score_path(name), artifact["test_score"])

  meta = {
    "name": name,
    "threshold": artifact["threshold"],
    "valid_metrics_05": artifact["valid_metrics_05"],
    "valid_metrics_opt": artifact["valid_metrics_opt"],
    "test_metrics_opt": artifact["test_metrics_opt"],
    "fit_seconds": artifact.get("fit_seconds", None),
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
  }

  with open(metadata_path(name), "w") as f:
    json.dump(meta, f, indent=2)

  artifact["threshold_df"].to_csv(
    METRIC_DIR / f"{name}_threshold_scan.csv",
    index=False,
  )

  score = artifact["test_score"]
  pred = (score >= artifact["threshold"]).astype(int)
  report = classification_report(y_test, pred, digits=4)

  with open(classification_report_path(name), "w") as f:
    f.write(report)

  print(f"Saved model artifact: {name}")


def load_artifact(name):
  if not model_path(name).exists() or not metadata_path(name).exists():
    return None

  try:
    model = joblib.load(model_path(name))

    with open(metadata_path(name), "r") as f:
      meta = json.load(f)

    if valid_score_path(name).exists():
      valid_score = np.load(valid_score_path(name))
    else:
      valid_score = get_score(model, X_valid)

    if test_score_path(name).exists():
      test_score = np.load(test_score_path(name))
    else:
      test_score = get_score(model, X_test)

    threshold_scan_file = METRIC_DIR / f"{name}_threshold_scan.csv"
    if threshold_scan_file.exists():
      threshold_df = pd.read_csv(threshold_scan_file)
      threshold = meta["threshold"]
    else:
      threshold, threshold_df = find_best_threshold(y_valid, valid_score)

    artifact = {
      "name": name,
      "model": model,
      "threshold": threshold,
      "threshold_df": threshold_df,
      "valid_score": valid_score,
      "test_score": test_score,
      "valid_metrics_05": meta["valid_metrics_05"],
      "valid_metrics_opt": meta["valid_metrics_opt"],
      "test_metrics_opt": meta["test_metrics_opt"],
    }

    print(f"Loaded saved model artifact: {name}")
    return artifact

  except Exception as exc:
    print(f"Failed to load {name}: {exc}")
    return None


def train_or_load_model(name, model_factory, fit_func):
  if (
    REUSE_SAVED_MODELS
    and not FORCE_RETRAIN_MODELS
    and model_path(name).exists()
  ):
    artifact = load_artifact(name)
    if artifact is not None:
      return artifact

  model = model_factory()
  artifact = fit_func(name, model)
  save_artifact(artifact)

  return artifact


artifacts = {}
valid_rows = []
test_rows = []


def record_artifact(artifact):
  artifacts[artifact["name"]] = artifact

  valid_rows.append({
    **artifact["valid_metrics_05"],
    "split": "valid",
    "setting": "threshold_0.5",
  })

  valid_rows.append({
    **artifact["valid_metrics_opt"],
    "split": "valid",
    "setting": "threshold_optimized",
  })

  test_rows.append({
    **artifact["test_metrics_opt"],
    "split": "test",
    "setting": "threshold_optimized",
  })

  pd.DataFrame(valid_rows).to_csv(
    METRIC_DIR / "summary_valid_partial.csv",
    index=False,
  )
  pd.DataFrame(test_rows).to_csv(
    METRIC_DIR / "summary_test_partial.csv",
    index=False,
  )


## 6. PyTorch models


In [15]:
if HAS_TORCH:
  import torch
  import torch.nn as nn
  from torch.utils.data import DataLoader, TensorDataset

  torch.manual_seed(RANDOM_STATE)
  torch.set_num_threads(TORCH_NUM_THREADS)

  try:
    torch.set_num_interop_threads(1)
  except RuntimeError:
    pass

  if FORCE_TORCH_CPU:
    DEVICE = torch.device("cpu")
  elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
  elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
  else:
    DEVICE = torch.device("cpu")

  print("PyTorch device:", DEVICE)
  print("PyTorch num threads:", torch.get_num_threads())
else:
  print("Torch not available.")


PyTorch device: cpu
PyTorch num threads: 1


In [16]:
def sample_rows_for_nn(X, y, max_rows, seed=RANDOM_STATE):
  if max_rows is None or len(X) <= max_rows:
    return X, y

  sampled_idx, _ = train_test_split(
    X.index,
    train_size=max_rows,
    random_state=seed,
    stratify=y,
  )

  return X.loc[sampled_idx], y.loc[sampled_idx]


def fit_and_score_nn_model(name, model, max_rows):
  X_train_nn, y_train_nn = sample_rows_for_nn(
    X_train,
    y_train,
    max_rows,
    seed=RANDOM_STATE,
  )

  print("=" * 80)
  print(f"Training {name}")
  print(f"Full train rows: {len(X_train):,}")
  print(f"NN train rows:   {len(X_train_nn):,}")
  print(f"Valid rows:      {len(X_valid):,}")
  print(f"Test rows:       {len(X_test):,}")
  print(f"Target rate in NN train: {y_train_nn.mean():.4f}")
  print("=" * 80, flush=True)

  start = time.time()

  print("[1/4] Fitting preprocessing + neural network...", flush=True)
  model.fit(X_train_nn, y_train_nn)
  fit_seconds = time.time() - start
  print(f"Fit finished in {fit_seconds:.1f} seconds", flush=True)

  print("[2/4] Predicting validation scores...", flush=True)
  valid_score = get_score(model, X_valid)

  print("[3/4] Optimizing threshold...", flush=True)
  threshold, threshold_df = find_best_threshold(y_valid, valid_score)

  valid_metrics_05 = evaluate_scores(
    name,
    y_valid,
    valid_score,
    threshold=0.5,
  )

  valid_metrics_opt = evaluate_scores(
    name,
    y_valid,
    valid_score,
    threshold=threshold,
  )

  print("[4/4] Predicting test scores...", flush=True)
  test_score = get_score(model, X_test)

  test_metrics_opt = evaluate_scores(
    name,
    y_test,
    test_score,
    threshold=threshold,
  )

  for row in [valid_metrics_05, valid_metrics_opt, test_metrics_opt]:
    row["fit_seconds"] = fit_seconds
    row["nn_train_rows"] = len(X_train_nn)

  return {
    "name": name,
    "model": model,
    "threshold": threshold,
    "threshold_df": threshold_df,
    "valid_score": valid_score,
    "test_score": test_score,
    "valid_metrics_05": valid_metrics_05,
    "valid_metrics_opt": valid_metrics_opt,
    "test_metrics_opt": test_metrics_opt,
    "fit_seconds": fit_seconds,
  }


In [17]:
if HAS_TORCH:
  def to_dense_float32(x):
    if hasattr(x, "toarray"):
      x = x.toarray()
    return np.asarray(x, dtype=np.float32)


  class TorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, depth=3, dropout=0.15):
      super().__init__()

      layers = []
      in_dim = input_dim

      for _ in range(depth):
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        in_dim = hidden_dim

      layers.append(nn.Linear(in_dim, 1))
      self.net = nn.Sequential(*layers)

    def forward(self, x):
      return self.net(x).squeeze(-1)


  class NumericTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
      super().__init__()
      self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
      self.bias = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x):
      return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


  class FTTransformerLite(nn.Module):
    def __init__(
      self,
      n_features,
      d_token=16,
      n_heads=2,
      n_layers=1,
      dropout=0.15,
    ):
      super().__init__()

      self.tokenizer = NumericTokenizer(n_features, d_token)
      self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

      encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_token,
        nhead=n_heads,
        dim_feedforward=4 * d_token,
        dropout=dropout,
        batch_first=True,
        activation="gelu",
      )

      self.encoder = nn.TransformerEncoder(
        encoder_layer,
        num_layers=n_layers,
      )

      self.head = nn.Sequential(
        nn.LayerNorm(d_token),
        nn.Linear(d_token, 1),
      )

    def forward(self, x):
      tokens = self.tokenizer(x)
      cls = self.cls.expand(x.shape[0], -1, -1)
      tokens = torch.cat([cls, tokens], dim=1)
      encoded = self.encoder(tokens)
      cls_out = encoded[:, 0, :]
      return self.head(cls_out).squeeze(-1)


  class TorchTabularClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
      self,
      model_type="mlp",
      hidden_dim=128,
      depth=3,
      dropout=0.15,
      lr=1e-3,
      weight_decay=1e-4,
      batch_size=4096,
      max_epochs=12,
      patience=4,
      d_token=16,
      n_heads=2,
      n_layers=1,
      random_state=RANDOM_STATE,
      verbose=False,
    ):
      self.model_type = model_type
      self.hidden_dim = hidden_dim
      self.depth = depth
      self.dropout = dropout
      self.lr = lr
      self.weight_decay = weight_decay
      self.batch_size = batch_size
      self.max_epochs = max_epochs
      self.patience = patience
      self.d_token = d_token
      self.n_heads = n_heads
      self.n_layers = n_layers
      self.random_state = random_state
      self.verbose = verbose

    def _make_model(self, input_dim):
      if self.model_type == "mlp":
        return TorchMLP(
          input_dim=input_dim,
          hidden_dim=self.hidden_dim,
          depth=self.depth,
          dropout=self.dropout,
        )

      if self.model_type == "ft_transformer":
        return FTTransformerLite(
          n_features=input_dim,
          d_token=self.d_token,
          n_heads=self.n_heads,
          n_layers=self.n_layers,
          dropout=self.dropout,
        )

      raise ValueError(f"Unknown model_type: {self.model_type}")

    def fit(self, X, y):
      torch.manual_seed(self.random_state)

      print("  [NN] converting preprocessed matrix to dense float32...", flush=True)
      X = to_dense_float32(X)
      y = np.asarray(y, dtype=np.float32)

      print(f"  [NN] dense matrix shape: {X.shape}", flush=True)
      print(f"  [NN] dense matrix size: {X.nbytes / 1e9:.2f} GB", flush=True)

      X_train_arr, X_val_arr, y_train_arr, y_val_arr = train_test_split(
        X,
        y,
        test_size=0.15,
        random_state=self.random_state,
        stratify=y,
      )

      if len(X_val_arr) > NN_VALID_INTERNAL_MAX_ROWS:
        val_idx, _ = train_test_split(
          np.arange(len(X_val_arr)),
          train_size=NN_VALID_INTERNAL_MAX_ROWS,
          random_state=self.random_state,
          stratify=y_val_arr,
        )

        X_val_arr = X_val_arr[val_idx]
        y_val_arr = y_val_arr[val_idx]

      print(f"  [NN] internal train rows: {len(X_train_arr):,}", flush=True)
      print(f"  [NN] internal valid rows: {len(X_val_arr):,}", flush=True)

      self.classes_ = np.array([0, 1])
      self.input_dim_ = X.shape[1]
      self.model_ = self._make_model(self.input_dim_).to(DEVICE)

      pos = max(float((y_train_arr == 1).sum()), 1.0)
      neg = max(float((y_train_arr == 0).sum()), 1.0)
      pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(DEVICE)

      loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
      optimizer = torch.optim.AdamW(
        self.model_.parameters(),
        lr=self.lr,
        weight_decay=self.weight_decay,
      )

      train_ds = TensorDataset(
        torch.tensor(X_train_arr),
        torch.tensor(y_train_arr),
      )
      val_x = torch.tensor(X_val_arr).to(DEVICE)

      train_loader = DataLoader(
        train_ds,
        batch_size=self.batch_size,
        shuffle=True,
        drop_last=False,
      )

      best_ap = -np.inf
      best_state = None
      bad_epochs = 0
      self.history_ = []

      n_batches = len(train_loader)

      for epoch in range(self.max_epochs):
        epoch_start = time.time()
        self.model_.train()
        train_loss_sum = 0.0
        n_seen = 0

        for batch_i, (xb, yb) in enumerate(train_loader):
          xb = xb.to(DEVICE)
          yb = yb.to(DEVICE)

          optimizer.zero_grad()
          logits = self.model_(xb)
          loss = loss_fn(logits, yb)
          loss.backward()
          optimizer.step()

          train_loss_sum += float(loss.item()) * len(xb)
          n_seen += len(xb)

          should_print = (
            batch_i == 0
            or (batch_i + 1) % NN_PROGRESS_EVERY_N_BATCHES == 0
            or (batch_i + 1) == n_batches
          )

          if self.verbose and should_print:
            print(
              f"  epoch {epoch + 1:03d}/{self.max_epochs:03d} "
              f"batch {batch_i + 1:04d}/{n_batches:04d} "
              f"loss={float(loss.item()):.4f}",
              flush=True,
            )

        self.model_.eval()
        with torch.no_grad():
          val_logits = self.model_(val_x)
          val_prob = torch.sigmoid(val_logits).cpu().numpy()

        val_ap = average_precision_score(y_val_arr, val_prob)
        train_loss = train_loss_sum / max(n_seen, 1)
        epoch_seconds = time.time() - epoch_start

        self.history_.append({
          "epoch": epoch,
          "train_loss": train_loss,
          "val_avg_precision": val_ap,
          "epoch_seconds": epoch_seconds,
        })

        if self.verbose:
          print(
            f"  epoch={epoch:03d} "
            f"train_loss={train_loss:.4f} "
            f"val_ap={val_ap:.4f} "
            f"time={epoch_seconds:.1f}s",
            flush=True,
          )

        if val_ap > best_ap:
          best_ap = val_ap
          best_state = {
            k: v.detach().cpu().clone()
            for k, v in self.model_.state_dict().items()
          }
          bad_epochs = 0
        else:
          bad_epochs += 1

        if bad_epochs >= self.patience:
          print("  [NN] early stopping triggered.", flush=True)
          break

      if best_state is not None:
        self.model_.load_state_dict(best_state)

      self.best_val_ap_ = best_ap
      return self

    def predict_proba(self, X):
      X = to_dense_float32(X)
      self.model_.eval()

      probs = []
      n = len(X)

      with torch.no_grad():
        for start in range(0, n, self.batch_size):
          stop = min(start + self.batch_size, n)
          xb = torch.tensor(X[start:stop]).to(DEVICE)
          logits = self.model_(xb)
          prob = torch.sigmoid(logits).cpu().numpy()
          probs.append(prob)

      p1 = np.concatenate(probs)
      p0 = 1 - p1

      return np.vstack([p0, p1]).T

    def predict(self, X):
      return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


## 7. Train/load baseline models


In [18]:
if RUN_BASELINES:
  baseline_specs = {
    "dummy_prior": lambda: Pipeline([
      ("preprocess", preprocess_tree),
      ("model", DummyClassifier(strategy="prior")),
    ]),
    "logreg_baseline": lambda: Pipeline([
      ("preprocess", preprocess_linear),
      ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
      )),
    ]),
  }

  for name, factory in baseline_specs.items():
    print(f"Starting {name}")
    artifact = train_or_load_model(
      name,
      factory,
      fit_and_score_sklearn_model,
    )
    record_artifact(artifact)


Starting dummy_prior
Saved model artifact: dummy_prior
Starting logreg_baseline
Saved model artifact: logreg_baseline


## 8. Train/load XGBoost


In [19]:
if RUN_XGBOOST and HAS_XGBOOST:
  from xgboost import XGBClassifier

  scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()

  def make_xgb_baseline():
    return Pipeline([
      ("preprocess", preprocess_tree),
      ("model", XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.035,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
      )),
    ])

  artifact = train_or_load_model(
    "xgboost_baseline",
    make_xgb_baseline,
    fit_and_score_sklearn_model,
  )
  record_artifact(artifact)

elif RUN_XGBOOST:
  print("Skipping XGBoost because xgboost is not installed.")


Saved model artifact: xgboost_baseline


In [ ]:
if RUN_XGBOOST and HAS_XGBOOST and HAS_OPTUNA and RUN_HYPERPARAM_OPT:
  import optuna
  from xgboost import XGBClassifier

  scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()
  study_path = STUDY_DIR / "optuna_xgboost.db"

  def objective_xgb(trial):
    params = {
      "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
      "max_depth": trial.suggest_int("max_depth", 2, 9),
      "learning_rate": trial.suggest_float(
        "learning_rate",
        0.006,
        0.08,
        log=True,
      ),
      "subsample": trial.suggest_float("subsample", 0.55, 1.0),
      "colsample_bytree": trial.suggest_float(
        "colsample_bytree",
        0.55,
        1.0,
      ),
      "min_child_weight": trial.suggest_float(
        "min_child_weight",
        1.0,
        40.0,
        log=True,
      ),
      "reg_lambda": trial.suggest_float(
        "reg_lambda",
        0.1,
        50.0,
        log=True,
      ),
      "reg_alpha": trial.suggest_float(
        "reg_alpha",
        1e-5,
        10.0,
        log=True,
      ),
    }

    model = Pipeline([
      ("preprocess", preprocess_tree),
      ("model", XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
      )),
    ])

    model.fit(X_train, y_train)
    score = get_score(model, X_valid)

    return average_precision_score(y_valid, score)

  study_xgb = optuna.create_study(
    direction="maximize",
    study_name="xgboost",
    storage=f"sqlite:///{study_path}",
    load_if_exists=True,
  )

  completed = [
    t for t in study_xgb.trials
    if t.state.name == "COMPLETE"
  ]

  remaining = max(OPTUNA_TRIALS_XGB - len(completed), 0)
  print(f"XGBoost completed trials: {len(completed)}")
  print(f"XGBoost remaining trials: {remaining}")

  if remaining > 0:
    study_xgb.optimize(
      objective_xgb,
      n_trials=remaining,
      timeout=OPTUNA_TIMEOUT_SECONDS,
      show_progress_bar=True,
    )

  print("Best XGBoost AP:", study_xgb.best_value)
  display(study_xgb.best_params)

  def make_xgb_optimized():
    return Pipeline([
      ("preprocess", preprocess_tree),
      ("model", XGBClassifier(
        **study_xgb.best_params,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
      )),
    ])

  artifact = train_or_load_model(
    "xgboost_optimized",
    make_xgb_optimized,
    fit_and_score_sklearn_model,
  )
  record_artifact(artifact)


[I 2026-06-05 23:12:34,370] A new study created in RDB with name: xgboost


XGBoost completed trials: 0
XGBoost remaining trials: 30


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-05 23:16:16,079] Trial 0 finished with value: 0.05330229602303853 and parameters: {'n_estimators': 960, 'max_depth': 4, 'learning_rate': 0.01293974951547357, 'subsample': 0.99696959698263, 'colsample_bytree': 0.7135727944570315, 'min_child_weight': 34.86112782910263, 'reg_lambda': 11.021127321586388, 'reg_alpha': 0.00023962088593174394}. Best is trial 0 with value: 0.05330229602303853.
[I 2026-06-05 23:19:48,323] Trial 1 finished with value: 0.05247611061783464 and parameters: {'n_estimators': 447, 'max_depth': 9, 'learning_rate': 0.027895225285790037, 'subsample': 0.8350581660917687, 'colsample_bytree': 0.664134416710468, 'min_child_weight': 1.6782572467707733, 'reg_lambda': 1.201128202479992, 'reg_alpha': 1.7597155112822385}. Best is trial 0 with value: 0.05330229602303853.
[I 2026-06-05 23:21:50,460] Trial 2 finished with value: 0.05376794211469013 and parameters: {'n_estimators': 341, 'max_depth': 4, 'learning_rate': 0.034362097002877574, 'subsample': 0.594626818675032, 

{'n_estimators': 663,
 'max_depth': 5,
 'learning_rate': 0.06250834889207496,
 'subsample': 0.9050162460512521,
 'colsample_bytree': 0.961469732408737,
 'min_child_weight': 6.382471230617324,
 'reg_lambda': 1.6281692815001405,
 'reg_alpha': 0.03945471329875304}

Saved model artifact: xgboost_optimized


: 

## 9. Train/load LightGBM


In [ ]:
if RUN_LIGHTGBM and HAS_LIGHTGBM:
  from lightgbm import LGBMClassifier

  scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()

  def make_lgbm_baseline():
    return Pipeline([
      ("preprocess", preprocess_tree),
      ("model", LGBMClassifier(
        n_estimators=900,
        num_leaves=63,
        learning_rate=0.025,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=100,
        reg_lambda=3.0,
        objective="binary",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
      )),
    ])

  artifact = train_or_load_model(
    "lightgbm_baseline",
    make_lgbm_baseline,
    fit_and_score_sklearn_model,
  )
  record_artifact(artifact)

elif RUN_LIGHTGBM:
  print("Skipping LightGBM because lightgbm is not installed.")


In [ ]:
if RUN_LIGHTGBM and HAS_LIGHTGBM and HAS_OPTUNA and RUN_HYPERPARAM_OPT:
  import optuna
  from lightgbm import LGBMClassifier

  scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()
  study_path = STUDY_DIR / "optuna_lightgbm.db"

  def objective_lgbm(trial):
    params = {
      "n_estimators": trial.suggest_int("n_estimators", 400, 1500),
      "num_leaves": trial.suggest_int("num_leaves", 31, 255),
      "learning_rate": trial.suggest_float(
        "learning_rate",
        0.005,
        0.07,
        log=True,
      ),
      "subsample": trial.suggest_float("subsample", 0.55, 1.0),
      "colsample_bytree": trial.suggest_float(
        "colsample_bytree",
        0.55,
        1.0,
      ),
      "min_child_samples": trial.suggest_int("min_child_samples", 30, 300),
      "reg_lambda": trial.suggest_float(
        "reg_lambda",
        0.1,
        50.0,
        log=True,
      ),
      "reg_alpha": trial.suggest_float(
        "reg_alpha",
        1e-5,
        10.0,
        log=True,
      ),
    }

    model = Pipeline([
      ("preprocess", preprocess_tree),
      ("model", LGBMClassifier(
        **params,
        objective="binary",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
      )),
    ])

    model.fit(X_train, y_train)
    score = get_score(model, X_valid)

    return average_precision_score(y_valid, score)

  study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lightgbm",
    storage=f"sqlite:///{study_path}",
    load_if_exists=True,
  )

  completed = [
    t for t in study_lgbm.trials
    if t.state.name == "COMPLETE"
  ]

  remaining = max(OPTUNA_TRIALS_LGBM - len(completed), 0)
  print(f"LightGBM completed trials: {len(completed)}")
  print(f"LightGBM remaining trials: {remaining}")

  if remaining > 0:
    study_lgbm.optimize(
      objective_lgbm,
      n_trials=remaining,
      timeout=OPTUNA_TIMEOUT_SECONDS,
      show_progress_bar=True,
    )

  print("Best LightGBM AP:", study_lgbm.best_value)
  display(study_lgbm.best_params)

  def make_lgbm_optimized():
    return Pipeline([
      ("preprocess", preprocess_tree),
      ("model", LGBMClassifier(
        **study_lgbm.best_params,
        objective="binary",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
      )),
    ])

  artifact = train_or_load_model(
    "lightgbm_optimized",
    make_lgbm_optimized,
    fit_and_score_sklearn_model,
  )
  record_artifact(artifact)


## 10. Train/load PyTorch MLP


In [ ]:
if RUN_MLP and HAS_TORCH:
  def make_mlp_baseline():
    return Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="mlp",
        hidden_dim=256,
        depth=4,
        dropout=0.15,
        lr=NN_LEARNING_RATE,
        weight_decay=1e-4,
        batch_size=NN_BATCH_SIZE,
        max_epochs=NN_MAX_EPOCHS,
        patience=NN_PATIENCE,
        random_state=RANDOM_STATE,
        verbose=True,
      )),
    ])

  artifact = train_or_load_model(
    "pytorch_mlp_baseline",
    make_mlp_baseline,
    lambda name, model: fit_and_score_nn_model(
      name,
      model,
      NN_TRAIN_MAX_ROWS,
    ),
  )
  record_artifact(artifact)


In [ ]:
if RUN_MLP and HAS_TORCH and HAS_OPTUNA and RUN_HYPERPARAM_OPT:
  import optuna

  study_path = STUDY_DIR / "optuna_pytorch_mlp.db"

  X_train_mlp_opt, y_train_mlp_opt = sample_rows_for_nn(
    X_train,
    y_train,
    min(NN_TRAIN_MAX_ROWS, 200_000),
    seed=RANDOM_STATE,
  )

  def objective_mlp(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 384])
    depth = trial.suggest_int("depth", 2, 5)
    dropout = trial.suggest_float("dropout", 0.05, 0.35)
    lr = trial.suggest_float("lr", 2e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float(
      "weight_decay",
      1e-6,
      1e-2,
      log=True,
    )

    model = Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="mlp",
        hidden_dim=hidden_dim,
        depth=depth,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=NN_BATCH_SIZE,
        max_epochs=max(5, NN_MAX_EPOCHS // 2),
        patience=2,
        random_state=RANDOM_STATE + trial.number,
        verbose=False,
      )),
    ])

    model.fit(X_train_mlp_opt, y_train_mlp_opt)
    score = get_score(model, X_valid)

    return average_precision_score(y_valid, score)

  study_mlp = optuna.create_study(
    direction="maximize",
    study_name="pytorch_mlp",
    storage=f"sqlite:///{study_path}",
    load_if_exists=True,
  )

  completed = [
    t for t in study_mlp.trials
    if t.state.name == "COMPLETE"
  ]

  remaining = max(OPTUNA_TRIALS_MLP - len(completed), 0)
  print(f"MLP completed trials: {len(completed)}")
  print(f"MLP remaining trials: {remaining}")

  if remaining > 0:
    study_mlp.optimize(
      objective_mlp,
      n_trials=remaining,
      timeout=OPTUNA_TIMEOUT_SECONDS,
      show_progress_bar=True,
    )

  print("Best MLP AP:", study_mlp.best_value)
  display(study_mlp.best_params)

  p = study_mlp.best_params

  def make_mlp_optimized():
    return Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="mlp",
        hidden_dim=p["hidden_dim"],
        depth=p["depth"],
        dropout=p["dropout"],
        lr=p["lr"],
        weight_decay=p["weight_decay"],
        batch_size=NN_BATCH_SIZE,
        max_epochs=NN_MAX_EPOCHS,
        patience=NN_PATIENCE,
        random_state=RANDOM_STATE,
        verbose=True,
      )),
    ])

  artifact = train_or_load_model(
    "pytorch_mlp_optimized",
    make_mlp_optimized,
    lambda name, model: fit_and_score_nn_model(
      name,
      model,
      NN_TRAIN_MAX_ROWS,
    ),
  )
  record_artifact(artifact)


## 11. Train/load FT-Transformer-lite


In [ ]:
if RUN_FT_TRANSFORMER and HAS_TORCH:
  def make_ft_baseline():
    return Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="ft_transformer",
        d_token=16,
        n_heads=2,
        n_layers=1,
        dropout=0.15,
        lr=NN_LEARNING_RATE,
        weight_decay=1e-4,
        batch_size=FT_BATCH_SIZE,
        max_epochs=FT_MAX_EPOCHS,
        patience=FT_PATIENCE,
        random_state=RANDOM_STATE,
        verbose=True,
      )),
    ])

  artifact = train_or_load_model(
    "ft_transformer_baseline",
    make_ft_baseline,
    lambda name, model: fit_and_score_nn_model(
      name,
      model,
      FT_TRAIN_MAX_ROWS,
    ),
  )
  record_artifact(artifact)


In [ ]:
if (
  RUN_FT_TRANSFORMER
  and HAS_TORCH
  and HAS_OPTUNA
  and RUN_HYPERPARAM_OPT
):
  import optuna

  study_path = STUDY_DIR / "optuna_ft_transformer.db"

  X_train_ft_opt, y_train_ft_opt = sample_rows_for_nn(
    X_train,
    y_train,
    min(FT_TRAIN_MAX_ROWS, 50_000),
    seed=RANDOM_STATE,
  )

  def objective_ft(trial):
    trial_start = time.time()

    d_token = trial.suggest_categorical("d_token", [8, 16, 24])
    n_heads_choices = [
      h for h in [1, 2, 4]
      if d_token % h == 0
    ]
    n_heads = trial.suggest_categorical("n_heads", n_heads_choices)
    n_layers = trial.suggest_int("n_layers", 1, 2)
    dropout = trial.suggest_float("dropout", 0.05, 0.30)
    lr = trial.suggest_float("lr", 2e-4, 2e-3, log=True)
    weight_decay = trial.suggest_float(
      "weight_decay",
      1e-6,
      1e-2,
      log=True,
    )

    print(
      f"FT trial {trial.number}: "
      f"d_token={d_token}, n_heads={n_heads}, n_layers={n_layers}",
      flush=True,
    )

    model = Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="ft_transformer",
        d_token=d_token,
        n_heads=n_heads,
        n_layers=n_layers,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=FT_BATCH_SIZE,
        max_epochs=max(3, FT_MAX_EPOCHS // 2),
        patience=1,
        random_state=RANDOM_STATE + trial.number,
        verbose=True,
      )),
    ])

    model.fit(X_train_ft_opt, y_train_ft_opt)
    score = get_score(model, X_valid)
    ap = average_precision_score(y_valid, score)

    print(
      f"FT trial {trial.number} AP={ap:.6f} "
      f"time={time.time() - trial_start:.1f}s",
      flush=True,
    )

    return ap

  study_ft = optuna.create_study(
    direction="maximize",
    study_name="ft_transformer",
    storage=f"sqlite:///{study_path}",
    load_if_exists=True,
  )

  completed = [
    t for t in study_ft.trials
    if t.state.name == "COMPLETE"
  ]

  remaining = max(OPTUNA_TRIALS_FT - len(completed), 0)
  print(f"FT completed trials: {len(completed)}")
  print(f"FT remaining trials: {remaining}")

  if remaining > 0:
    study_ft.optimize(
      objective_ft,
      n_trials=remaining,
      timeout=OPTUNA_TIMEOUT_SECONDS,
      show_progress_bar=True,
    )

  print("Best FT AP:", study_ft.best_value)
  display(study_ft.best_params)

  p = study_ft.best_params

  def make_ft_optimized():
    return Pipeline([
      ("preprocess", preprocess_nn),
      ("model", TorchTabularClassifier(
        model_type="ft_transformer",
        d_token=p["d_token"],
        n_heads=p["n_heads"],
        n_layers=p["n_layers"],
        dropout=p["dropout"],
        lr=p["lr"],
        weight_decay=p["weight_decay"],
        batch_size=FT_BATCH_SIZE,
        max_epochs=FT_MAX_EPOCHS,
        patience=FT_PATIENCE,
        random_state=RANDOM_STATE,
        verbose=True,
      )),
    ])

  artifact = train_or_load_model(
    "ft_transformer_optimized",
    make_ft_optimized,
    lambda name, model: fit_and_score_nn_model(
      name,
      model,
      FT_TRAIN_MAX_ROWS,
    ),
  )
  record_artifact(artifact)


## 12. Metrics, plots, and summaries


In [ ]:
summary_valid = pd.DataFrame(valid_rows).drop_duplicates(
  subset=["model", "setting"],
  keep="last",
)

summary_test = pd.DataFrame(test_rows).drop_duplicates(
  subset=["model", "setting"],
  keep="last",
)

summary_valid.to_csv(METRIC_DIR / "summary_valid.csv", index=False)
summary_test.to_csv(METRIC_DIR / "summary_test.csv", index=False)

display(summary_valid.sort_values("avg_precision", ascending=False))
display(summary_test.sort_values("avg_precision", ascending=False))


In [ ]:
def save_current_fig(filename):
  path = PLOT_DIR / filename
  plt.tight_layout()
  plt.savefig(path, dpi=180, bbox_inches="tight")
  print("Saved plot:", path)


for metric in ["avg_precision", "roc_auc", "f1", "balanced_acc"]:
  plot_df = summary_test.sort_values(metric, ascending=True)

  fig, ax = plt.subplots(figsize=(9, 5))
  ax.barh(plot_df["model"], plot_df[metric])
  ax.set_xlabel(f"Test {metric}")
  ax.set_title(f"Model comparison: {metric}")
  save_current_fig(f"model_comparison_{metric}.png")
  plt.show()


In [ ]:
overlay_artifacts = {
  name: artifact
  for name, artifact in artifacts.items()
  if name != "dummy_prior"
}

if len(overlay_artifacts) == 0:
  overlay_artifacts = artifacts

fig, ax = plt.subplots(figsize=(7, 6))
for name, artifact in overlay_artifacts.items():
  RocCurveDisplay.from_predictions(
    y_test,
    artifact["test_score"],
    name=name,
    ax=ax,
  )
ax.set_title("Test ROC curves")
save_current_fig("overlay_roc_curves.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
baseline_rate = y_test.mean()
ax.axhline(
  baseline_rate,
  linestyle="--",
  label=f"Positive rate = {baseline_rate:.3f}",
)
for name, artifact in overlay_artifacts.items():
  PrecisionRecallDisplay.from_predictions(
    y_test,
    artifact["test_score"],
    name=name,
    ax=ax,
  )
ax.set_title("Test precision-recall curves")
save_current_fig("overlay_precision_recall_curves.png")
plt.show()


In [ ]:
for name, artifact in artifacts.items():
  score = artifact["test_score"]
  pred = (score >= artifact["threshold"]).astype(int)

  disp = ConfusionMatrixDisplay(confusion_matrix(y_test, pred))
  disp.plot()
  plt.title(f"Test confusion matrix: {name}")
  save_current_fig(f"confusion_matrix_{name}.png")
  plt.show()


## 13. Permutation importance with checkpointing


In [ ]:
def importance_file(name):
  return IMPORTANCE_DIR / f"permutation_importance_{name}.csv"


importance_results = {}

if RUN_PERMUTATION_IMPORTANCE:
  small_n = min(PERM_IMPORTANCE_MAX_ROWS, len(X_valid))

  X_imp = X_valid.sample(small_n, random_state=RANDOM_STATE)
  y_imp = y_valid.loc[X_imp.index]

  for name, artifact in artifacts.items():
    if name == "dummy_prior":
      continue

    file_path = importance_file(name)

    if file_path.exists() and REUSE_SAVED_MODELS:
      print(f"Loading permutation importance: {name}")
      importance_df = pd.read_csv(file_path)
      importance_results[name] = importance_df
      continue

    print(f"Permutation importance: {name}")

    perm = permutation_importance(
      artifact["model"],
      X_imp,
      y_imp,
      n_repeats=PERM_IMPORTANCE_REPEATS,
      scoring="average_precision",
      random_state=RANDOM_STATE,
      n_jobs=1,
    )

    importance_df = pd.DataFrame({
      "model": name,
      "feature": feature_cols,
      "importance_mean": perm.importances_mean,
      "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    importance_df.to_csv(file_path, index=False)
    importance_results[name] = importance_df

    display(importance_df.head(20))

else:
  print("Skipping permutation importance.")


In [ ]:
if len(importance_results) > 0:
  for name, importance_df in importance_results.items():
    plot_df = importance_df.head(20).iloc[::-1]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(plot_df["feature"], plot_df["importance_mean"])
    ax.set_xlabel("Decrease in validation average precision")
    ax.set_title(f"Permutation importance: {name}")
    save_current_fig(f"permutation_importance_{name}.png")
    plt.show()

  all_importance = pd.concat(importance_results.values(), ignore_index=True)
  all_importance.to_csv(
    IMPORTANCE_DIR / "permutation_importance_all_models.csv",
    index=False,
  )

  normed = all_importance.copy()

  def normalize_group(s):
    denom = s.abs().sum()
    if denom == 0:
      return s
    return s / denom

  normed["importance_norm"] = (
    normed
    .groupby("model")["importance_mean"]
    .transform(normalize_group)
  )

  feature_summary = (
    normed
    .groupby("feature", as_index=False)["importance_norm"]
    .mean()
    .sort_values("importance_norm", ascending=False)
  )

  feature_summary.to_csv(
    IMPORTANCE_DIR / "permutation_importance_feature_summary.csv",
    index=False,
  )

  display(feature_summary)

  plot_df = feature_summary.head(20).iloc[::-1]

  fig, ax = plt.subplots(figsize=(8, 6))
  ax.barh(plot_df["feature"], plot_df["importance_norm"])
  ax.set_xlabel("Mean normalized permutation importance")
  ax.set_title("Cross-model average feature importance")
  save_current_fig("permutation_importance_cross_model_summary.png")
  plt.show()


## 14. Best model export


In [ ]:
if len(summary_test) > 0:
  best_row = summary_test.sort_values("avg_precision", ascending=False).iloc[0]
  best_model_name = best_row["model"]

  with open(RUN_DIR / "best_model_name.txt", "w") as f:
    f.write(str(best_model_name))

  best_summary = {
    "best_model_name": str(best_model_name),
    "selection_metric": "test_avg_precision",
    "test_metrics": best_row.to_dict(),
    "model_file": str(model_path(best_model_name)),
  }

  with open(RUN_DIR / "best_model_summary.json", "w") as f:
    json.dump(best_summary, f, indent=2)

  print("Best model:", best_model_name)
  display(pd.DataFrame([best_row]))
  print("Best model file:", model_path(best_model_name))
else:
  print("No test summary rows found.")


## Done

The important outputs are in:

```text
scratch/overnight-runs/blunder-prediction-overnight/
```

If the notebook crashes and you rerun it with `REUSE_SAVED_MODELS = True`, it
will load completed models and continue from the missing pieces.
